In [ ]:
import marimo as mo
from pathlib import Path
import os

import pandas as pd
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, r2_score, classification_report

from PIL import Image
import torchvision.models as models
import torchvision.transforms as transforms

In [ ]:
WDIR = 'data/cv-lego/'
collections = ['harry-potter', 'jurassic-world', 'marvel', 'star-wars']
TDIR = Path('data/cv-lego/test')
INDEXES = Path('data/cv-lego/index.csv')

In [ ]:
for theme in collections:
    count_photo = 0
    dirs = os.walk(WDIR + theme)
    for dir in dirs:
        count_photo += len(dir[2])
    print(f'Коллекция {theme}, кол-во фото {count_photo}')
print()
print(f'Коллекция test, кол-во фото {len(os.listdir(TDIR))}')

In [ ]:
indexes = pd.read_csv(INDEXES)

In [ ]:
indexes.head()

In [ ]:
print(indexes.isna().sum())

In [ ]:
indexes.class_id = indexes.class_id.map(lambda cl: cl - 1)

In [ ]:
classes = indexes.class_id.nunique()
classes

In [ ]:
# смотрим чтобы были такие же названия как и в папках
indexes.path.map(lambda name: name.split('/')[0]).unique()

In [ ]:
dataset = indexes.copy()

In [ ]:
dataset['vectorize'] = None

## Преобразовываем все картинки в вектора для дальнейшего построения CNN

In [ ]:
# model_ = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model_ = models.resnext50_32x4d(weights=models.ResNeXt50_32X4D_Weights.IMAGENET1K_V2)

In [ ]:
model = torch.nn.Sequential(*(
    list(model_.children())[:-1]
))

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
def img_to_vec(image_path):
    model.eval()
    img = Image.open(image_path).convert('RGB')
    img_tensor = preprocess(img)
    img_tensor = img_tensor.unsqueeze(0)

    with torch.no_grad():
        vector = model(img_tensor)

    return vector.squeeze().numpy()

In [ ]:
dataset['vectorize'] = dataset.path.map(lambda path: img_to_vec(WDIR + path))

In [ ]:
dataset.to_csv(WDIR + 'dataset.csv')

In [ ]:
dataset.head()

## Грузим датасет

In [ ]:
vectors_for_linreg = np.stack(dataset["vectorize"].values)   # shape: (N, 2048)
y = dataset["class_id"].values                   # shape: (N,)

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(vectors_for_linreg, y)

In [ ]:
vectors = dataset.vectorize
y_true = dataset.class_id

In [ ]:
t_x = torch.tensor(vectors)
t_y = torch.tensor(y_true)

In [ ]:
x_tensor_dataset = TensorDataset(t_x, t_y)

In [ ]:
x_tensor_dataset.tensors

In [ ]:
x_train_dl = DataLoader(x_tensor_dataset, batch_size=32, shuffle=True)

## Реализация модели

In [ ]:
# ps просто захотелось, так сначала бы начал с логистической регрессии, картинки бы так же преобразовал сначала

class CV(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.net = nn.Sequential(
            nn.Linear(2048, 38),
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
cnn = CV()
optim = torch.optim.AdamW(cnn.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()

In [ ]:
def train(model: nn.Module, optimizer, loss_func, epoch: int = 5):
    model.train()
    loss_mean = 0
    for epo in range(epoch):
        for xb, yb in x_train_dl:
            pred = model(xb)
            loss = loss_func(pred, yb)
            loss_mean += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        if epo % 10 == 0 or epo == epoch - 1:
            print(f'Эпоха: {epo}, ошибка: {np.mean(loss_mean)}')
            loss_mean = 0

In [ ]:
train(cnn, optim, loss_func, 300)

In [ ]:
test_df = pd.read_csv('data/cv-lego/test.csv')

In [ ]:
test_df['x'] = test_df.path.map(lambda img: img_to_vec(WDIR + img))

In [ ]:
test_df.class_id = test_df.class_id.map(lambda id: id - 1)

In [ ]:
test_df.head()

In [ ]:
test_x_t = torch.tensor(test_df.x)
test_y_t = torch.tensor(test_df.class_id)

In [ ]:
test_ds = TensorDataset(test_x_t, test_y_t)
test_dl = DataLoader(test_ds, batch_size=32)

In [ ]:
def model_testing(model: nn.Module, loader: torch.utils.data.DataLoader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for xb, yb in loader:
            outputs = model(xb)
            _, predicted = torch.max(outputs.data, 1)
            total += yb.size(0)
            correct += (predicted == yb).sum().item()
        accuracy = 100 * correct / total
    print(f'\nТочность на тестовой выборке: {accuracy:.2f}')
    return accuracy

In [ ]:
model_testing(cnn, test_dl)

In [ ]:
# Пробуем линейку

In [ ]:
linReg = LinearRegression()

In [ ]:
linReg.fit(X_train, y_train)

In [ ]:
y_pred_linReg = linReg.predict(X_val)

In [ ]:
mean_absolute_error(y_val, y_pred_linReg)

In [ ]:
r2_score(y_val, y_pred_linReg)

In [ ]:
logReg = LogisticRegression(max_iter=500).fit(X_train, y_train)

In [ ]:
y_pred_logReg = logReg.predict(X_val)

In [ ]:
print(classification_report(y_val, y_pred_logReg))